# Notebook to create the dataframe for the analysis of the data

In [6]:
import json
import geopandas as gpd
import pandas as pd
import sys
import os
sys.path.append(os.path.join(os.path.join(os.getcwd(), '../')))
from utilities.common_functions import create_historical_dataframe, clean_dataframe, create_continous_wind_dataframe, clean_continous_wind_dataframe
import numpy as np


In [7]:
gdf = gpd.read_file('../data/stations_files/stations_area_of_interest.geojson')
stations_names = gdf['station'].tolist()
years = range(2019, 2024)



# Historical data

## Create historical dataframe

In [ ]:
with open('../data/stations_files/no_historical_data_stations.json', 'r') as f:
    no_historical_data_stations = json.load(f)
    
# create array of empty dataframes
years_dataframes = []
for year in years:
    years_dataframes.append(pd.DataFrame())

In [ ]:
for ind, year in enumerate(years):
    for station_name in stations_names:
        if station_name not in no_historical_data_stations[str(year)]:   
            years_dataframes[ind] = create_historical_dataframe(str(station_name), year, years_dataframes[ind], gdf)
    
        



## Saving the csv

In [ ]:
for ind, year_dataframe in enumerate(years_dataframes):
    years_dataframes[ind] = clean_dataframe(year_dataframe)
    years_dataframes[ind].to_csv(f'../data/years_dataframe/{years[ind]}_data.csv')

## Loading the csv and creating a netcdf

In [ ]:
unwanted_values = [99.0, 999.0, 9999.0]
for ind, year in enumerate(years):
    year_dataframe = pd.read_csv(f'../data/years_dataframe/{year}_data.csv')  
    year_dataframe['station_id'] = year_dataframe['station_id'].astype('str')
    year_dataframe['datetime'] = pd.to_datetime(year_dataframe['datetime'])
    year_dataframe['time'] = [t.astype('datetime64[h]') for t in year_dataframe.datetime.values]
    year_dataframe = year_dataframe[~year_dataframe[['WDIR', 'WSPD']].isin(unwanted_values).any(axis=1)]
    # Replace the specified values with NaN
    year_dataframe.replace(unwanted_values, np.nan, inplace=True)
    year_dataframe.loc[:, 'u_velocity'] = year_dataframe['WDIR'].apply(lambda x: np.cos(np.radians(270 - x))) * year_dataframe['WSPD']
    year_dataframe.loc[:, 'v_velocity'] = year_dataframe['WDIR'].apply(lambda x: np.sin(np.radians(270 - x))) * year_dataframe['WSPD']
    longitudes = year_dataframe.groupby('station_id').mean()['longitude']
    latitudes = year_dataframe.groupby('station_id').mean()['latitude']
    ds = year_dataframe.groupby(['station_id','time']).mean().drop(columns=['WDIR', 'WSPD', 'longitude', 'latitude', 'datetime']).to_xarray()   
    ds['latitude'] = (('station_id'), latitudes)
    ds['longitude'] = (('station_id'), longitudes)
    ds.to_netcdf(f'../data/years_dataframe/{year}_data.nc')


# Continous Wind


## Create wind dataframe

In [12]:
with open('../data/stations_files/failed_stations_continous_wind.json', 'r') as f:
    no_continous_wind_stations = json.load(f)

# create array of empty dataframes
continous_wind_dataframes = []
for year in years:
    continous_wind_dataframes.append(pd.DataFrame())

In [13]:
for ind, year in enumerate(years):
    for station_name in stations_names:
        if station_name not in no_continous_wind_stations[str(year)]:   
            continous_wind_dataframes[ind] = create_continous_wind_dataframe(str(station_name), year, continous_wind_dataframes[ind], gdf)

## Saving CSV

In [14]:
for ind, year_dataframe in enumerate(continous_wind_dataframes):
    continous_wind_dataframes[ind] = clean_continous_wind_dataframe(year_dataframe)
    continous_wind_dataframes[ind].to_csv(f'../data/years_dataframe/{years[ind]}_continous_wind_data.csv')

## Loading the csv and creating a netcdf

In [ ]:
unwanted_values = [99.0, 999.0, 9999.0]
for ind, year in enumerate(years):
    year_dataframe = pd.read_csv(f'../data/years_dataframe/{year}_continous_wind_data.csv')  
    year_dataframe['station_id'] = year_dataframe['station_id'].astype('str')
    year_dataframe['datetime'] = pd.to_datetime(year_dataframe['datetime'])
    year_dataframe['time'] = [t.astype('datetime64[h]') for t in year_dataframe.datetime.values]
    year_dataframe = year_dataframe[~year_dataframe[['WDIR', 'WSPD']].isin(unwanted_values).any(axis=1)]
    # Replace the specified values with NaN
    year_dataframe.replace(unwanted_values, np.nan, inplace=True)
    year_dataframe.loc[:, 'u_velocity'] = year_dataframe['WDIR'].apply(lambda x: np.cos(np.radians(270 - x))) * year_dataframe['WSPD']
    year_dataframe.loc[:, 'v_velocity'] = year_dataframe['WDIR'].apply(lambda x: np.sin(np.radians(270 - x))) * year_dataframe['WSPD']
    year_dataframe.loc[:, 'gust_u_velocity'] = year_dataframe['GDR'].apply(lambda x: np.cos(np.radians(270 - x))) * year_dataframe['GST']
    year_dataframe.loc[:, 'gust_v_velocity'] = year_dataframe['GDR'].apply(lambda x: np.sin(np.radians(270 - x))) * year_dataframe['GST']
    longitudes = year_dataframe.groupby('station_id').mean()['longitude']
    latitudes = year_dataframe.groupby('station_id').mean()['latitude']
    ds = year_dataframe.groupby(['station_id','time']).mean().drop(columns=['WDIR', 'WSPD', 'GDR', 'GST','longitude', 'latitude', 'datetime']).to_xarray()   
    ds['latitude'] = (('station_id'), latitudes)
    ds['longitude'] = (('station_id'), longitudes)
    ds.to_netcdf(f'../data/years_dataframe/{year}_continous_wind_data.nc')